# W10D2 — LangGraph State Machines & Conditional Edges

## Objective
Build and test a stateful LangGraph workflow using conditional edges, classification, routing, and human-in-the-loop interaction.

## Tasks
1. Build the three-node graph: `classify → route → respond`.
2. Implement conditional edges based on classification.
3. Test the graph with five different inputs.
4. Test human-in-the-loop pause, human input, and resume.
5. Save evaluation evidence and validation results.

## Technology Stack
- Python
- LangGraph `StateGraph`
- `TypedDict` for state
- `InMemorySaver` for checkpointing
- Jupyter Notebook

In [1]:
import sys
from importlib.metadata import version

print("Python executable:", sys.executable)
print("LangGraph version:", version("langgraph"))
print("LangChain Core version:", version("langchain-core"))

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

print("StateGraph: OK")
print("START/END: OK")
print("InMemorySaver: OK")

Python executable: c:\Users\ughan\CynarisInternship\AIML_PROJECT\.venv-w9d1\Scripts\python.exe
LangGraph version: 1.2.16
LangChain Core version: 1.6.13
StateGraph: OK
START/END: OK
InMemorySaver: OK


In [2]:
from typing import TypedDict


class AgentState(TypedDict):
    user_input: str
    category: str
    route: str
    response: str


print("State schema defined.")
print("State fields:", list(AgentState.__annotations__.keys()))

State schema defined.
State fields: ['user_input', 'category', 'route', 'response']


In [3]:
def classify_node(state: AgentState) -> dict:
    text = state["user_input"].lower()

    if any(word in text for word in ["error", "bug", "failed", "exception"]):
        category = "technical"
    elif any(word in text for word in ["hello", "hi", "hey", "good morning"]):
        category = "greeting"
    elif any(word in text for word in ["help", "how", "what", "explain", "why"]):
        category = "question"
    else:
        category = "general"

    return {"category": category}


# Test the classification node.
sample_state = {
    "user_input": "My Python code has an error",
    "category": "",
    "route": "",
    "response": "",
}

classification = classify_node(sample_state)

print("Input:", sample_state["user_input"])
print("Category:", classification["category"])

Input: My Python code has an error
Category: technical


In [4]:
def route_node(state: AgentState) -> dict:
    route_map = {
        "technical": "technical_support",
        "greeting": "friendly_greeting",
        "question": "answer_question",
        "general": "general_response",
    }

    category = state["category"]
    return {"route": route_map.get(category, "general_response")}


# Test routing.
routing_state = {
    "user_input": "My Python code has an error",
    "category": "technical",
    "route": "",
    "response": "",
}

routing_result = route_node(routing_state)

print("Category:", routing_state["category"])
print("Selected route:", routing_result["route"])

Category: technical
Selected route: technical_support


In [5]:
def respond_node(state: AgentState) -> dict:
    route = state["route"]
    user_input = state["user_input"]

    responses = {
        "technical_support": (
            f"I can help troubleshoot this technical issue: {user_input}"
        ),
        "friendly_greeting": "Hello! How can I help you today?",
        "answer_question": f"Let's work through your question: {user_input}",
        "general_response": f"Thanks for sharing. Tell me more about: {user_input}",
    }

    return {
        "response": responses.get(
            route, "I'm ready to help. Please provide more details."
        )
    }


# Test the response node.
response_state = {
    "user_input": "My Python code has an error",
    "category": "technical",
    "route": "technical_support",
    "response": "",
}

response_result = respond_node(response_state)

print("Generated response:")
print(response_result["response"])

Generated response:
I can help troubleshoot this technical issue: My Python code has an error


In [6]:
from langgraph.graph import StateGraph, START, END


def build_state_machine():
    builder = StateGraph(AgentState)

    # Register the three application nodes.
    builder.add_node("classify", classify_node)
    builder.add_node("route", route_node)
    builder.add_node("respond", respond_node)

    # Connect the graph start to classification.
    builder.add_edge(START, "classify")

    # Classification feeds the routing node.
    builder.add_edge("classify", "route")

    # Conditional edges choose the route based on state.
    builder.add_conditional_edges(
        "route",
        lambda state: state["route"],
        {
            "technical_support": "respond",
            "friendly_greeting": "respond",
            "answer_question": "respond",
            "general_response": "respond",
        },
    )

    builder.add_edge("respond", END)

    return builder.compile()


state_machine = build_state_machine()

print("State machine compiled successfully.")
print("Application nodes:", ["classify", "route", "respond"])

State machine compiled successfully.
Application nodes: ['classify', 'route', 'respond']


In [7]:
test_inputs = [
    "Hello there!",
    "My Python code has an error",
    "How does a robot use computer vision?",
    "Explain LangGraph to me",
    "Today I worked on my project",
]

test_results = []

for user_input in test_inputs:
    result = state_machine.invoke({
        "user_input": user_input,
        "category": "",
        "route": "",
        "response": "",
    })

    test_results.append({
        "input": user_input,
        "category": result["category"],
        "route": result["route"],
        "response": result["response"],
    })

expected_categories = [
    "greeting",
    "technical",
    "question",
    "question",
    "general",
]

for index, (result, expected) in enumerate(
    zip(test_results, expected_categories), start=1
):
    passed = (
        result["category"] == expected
        and bool(result["route"])
        and bool(result["response"])
    )
    print(
        f"Test {index}: {'PASS' if passed else 'FAIL'} | "
        f"Expected={expected} | "
        f"Actual={result['category']} | "
        f"Route={result['route']}"
    )

passed_count = sum(
    result["category"] == expected
    and bool(result["route"])
    and bool(result["response"])
    for result, expected in zip(test_results, expected_categories)
)

print(f"\nResult: {passed_count}/5 tests passed")

Test 1: PASS | Expected=greeting | Actual=greeting | Route=friendly_greeting
Test 2: PASS | Expected=technical | Actual=technical | Route=technical_support
Test 3: PASS | Expected=question | Actual=question | Route=answer_question
Test 4: PASS | Expected=question | Actual=question | Route=answer_question
Test 5: PASS | Expected=general | Actual=general | Route=general_response

Result: 5/5 tests passed


In [8]:
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver


def human_review_node(state: AgentState) -> dict:
    decision = interrupt({
        "message": "Human review required",
        "draft_response": state["response"],
        "question": "Approve this response or provide a replacement?",
    })

    if isinstance(decision, dict):
        final_response = decision.get("response", state["response"])
    else:
        final_response = str(decision)

    return {"response": final_response}


def build_human_review_graph():
    builder = StateGraph(AgentState)

    builder.add_node("classify", classify_node)
    builder.add_node("route", route_node)
    builder.add_node("respond", respond_node)
    builder.add_node("human_review", human_review_node)

    builder.add_edge(START, "classify")
    builder.add_edge("classify", "route")
    builder.add_conditional_edges(
        "route",
        lambda state: state["route"],
        {
            "technical_support": "respond",
            "friendly_greeting": "respond",
            "answer_question": "respond",
            "general_response": "respond",
        },
    )
    builder.add_edge("respond", "human_review")
    builder.add_edge("human_review", END)

    return builder.compile(checkpointer=InMemorySaver())


human_review_graph = build_human_review_graph()
print("Human-review state machine compiled:", human_review_graph is not None)

Human-review state machine compiled: True


In [9]:
review_config = {
    "configurable": {"thread_id": "w10d2-human-review-1"}
}

initial_state = {
    "user_input": "How does a robot use computer vision?",
    "category": "",
    "route": "",
    "response": "",
}

paused_result = human_review_graph.invoke(
    initial_state,
    config=review_config,
)

print("Graph paused for human review.")
print("Interrupt present:", bool(paused_result.get("__interrupt__")))

if paused_result.get("__interrupt__"):
    print("Review request:")
    print(paused_result["__interrupt__"][0].value)

Graph paused for human review.
Interrupt present: True
Review request:
{'message': 'Human review required', 'draft_response': "Let's work through your question: How does a robot use computer vision?", 'question': 'Approve this response or provide a replacement?'}


In [10]:
human_decision = {
    "response": (
        "Computer vision helps robots interpret camera images to detect "
        "objects, estimate positions, and navigate their environment."
    )
}

resumed_result = human_review_graph.invoke(
    Command(resume=human_decision),
    config=review_config,
)

print("Graph resumed successfully.")
print("Final response:")
print(resumed_result["response"])
print("Interrupt still pending:", bool(resumed_result.get("__interrupt__")))

Graph resumed successfully.
Final response:
Computer vision helps robots interpret camera images to detect objects, estimate positions, and navigate their environment.
Interrupt still pending: False


In [11]:
import json
from pathlib import Path

output_dir = Path("../outputs/W10D2")
output_dir.mkdir(parents=True, exist_ok=True)

evaluation = {
    "assignment": "W10D2 - LangGraph State Machines & Conditional Edges",
    "classification_tests": {
        "total": len(test_results),
        "passed": passed_count,
        "failed": len(test_results) - passed_count,
        "results": test_results,
    },
    "human_review": {
        "graph_compiled": True,
        "interrupt_detected": bool(paused_result.get("__interrupt__")),
        "resumed_successfully": bool(resumed_result["response"]),
        "interrupt_pending_after_resume": bool(
            resumed_result.get("__interrupt__")
        ),
        "final_response": resumed_result["response"],
    },
}

evaluation_path = output_dir / "evaluation_results.json"
evaluation_path.write_text(
    json.dumps(evaluation, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Evaluation saved:", evaluation_path.resolve())
print("Classification tests:", f"{passed_count}/{len(test_results)}")
print(
    "Human-review pause:",
    evaluation["human_review"]["interrupt_detected"],
)
print(
    "Human-review resume:",
    evaluation["human_review"]["resumed_successfully"],
)
print("Evidence file exists:", evaluation_path.exists())

Evaluation saved: C:\Users\ughan\CynarisInternship\AIML_PROJECT\outputs\W10D2\evaluation_results.json
Classification tests: 5/5
Human-review pause: True
Human-review resume: True
Evidence file exists: True


In [12]:
validation_report = """# W10D2 Validation Report

## Assignment
LangGraph State Machines & Conditional Edges

## Validation Results
- [x] Three-node graph compiled: classify -> route -> respond
- [x] Conditional routing configured
- [x] Five input classification tests passed (5/5)
- [x] Human-review graph compiled
- [x] Graph paused at the human-review interrupt
- [x] Human-provided response submitted
- [x] Graph resumed successfully
- [x] No interrupt remained after resume
- [x] Evaluation evidence saved to evaluation_results.json

## Evidence
- `evaluation_results.json` contains the five test results and human-review outcomes.

## Pending Workflow Checks
- [ ] CIA Full Stack Mentor Mode code review
- [ ] At least two CIA interactions logged
- [ ] At least two meaningful Git commits
- [ ] Branch pushed and pull request created
"""

validation_path = output_dir / "VALIDATION.md"
validation_path.write_text(validation_report, encoding="utf-8")

print("Validation report saved:", validation_path.resolve())
print("Report exists:", validation_path.exists())

Validation report saved: C:\Users\ughan\CynarisInternship\AIML_PROJECT\outputs\W10D2\VALIDATION.md
Report exists: True


In [13]:
import json

evaluation_path = output_dir / "evaluation_results.json"
validation_path = output_dir / "VALIDATION.md"

with evaluation_path.open("r", encoding="utf-8") as file:
    saved_evaluation = json.load(file)

assert saved_evaluation["classification_tests"]["passed"] == 5
assert saved_evaluation["classification_tests"]["failed"] == 0
assert saved_evaluation["human_review"]["interrupt_detected"] is True
assert saved_evaluation["human_review"]["resumed_successfully"] is True
assert saved_evaluation["human_review"]["interrupt_pending_after_resume"] is False
assert validation_path.exists()

print("Saved JSON: valid")
print("Classification tests: 5/5 passed")
print("Human-review pause/resume: passed")
print("Validation report: present")
print("Evidence verification: PASS")

Saved JSON: valid
Classification tests: 5/5 passed
Human-review pause/resume: passed
Validation report: present
Evidence verification: PASS
